# CardioScope, Part 2: from model to clinical tool

**BIOVANCE 2026, École Polytechnique de Sousse**. Workshop *Deep Learning for Cardiovascular Risk Detection*.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nevermind78/cardioscope-workshop/blob/main/Part2_Model_to_Clinic.ipynb)

This morning we built a beat classifier and evaluated it honestly. A good score is not enough for a clinical tool:

| Lab | Question |
|---|---|
| 5 | Can we trust its probabilities? Does it know when it does not know? |
| 6 | Why does it take a decision? |
| 7 | How do beat labels become a patient-level risk profile? |
| Deploy | Can a clinician use it without writing code? |

In [ ]:
# Setup: run this cell first (about 1 minute on Colab)
%matplotlib inline
import os, sys, subprocess
os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")  # avoid an OpenMP crash (MKL + PyTorch both bundle libiomp5md)
REPO_URL = "https://github.com/nevermind78/cardioscope-workshop.git"   # set once by the instructor
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    if not os.path.isdir("/content/cardioscope-workshop"):
        if "YOUR-ACCOUNT" not in REPO_URL:
            subprocess.run(["git", "clone", "-q", REPO_URL, "/content/cardioscope-workshop"], check=True)
        else:                                   # no GitHub repository: upload the workshop ZIP
            from google.colab import files
            print("Upload cardioscope-workshop.zip (given by the instructor)")
            zip_name = next(iter(files.upload()))
            subprocess.run(["unzip", "-q", "-o", zip_name, "-d", "/content"], check=True)
    os.chdir("/content/cardioscope-workshop")
    if os.getcwd() not in sys.path:                 # os.chdir alone doesn't update sys.path on Colab
        sys.path.insert(0, os.getcwd())
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-colab.txt"], check=True)

import numpy as np, pandas as pd, torch
from cardioscope import config as C, data as D, viz
from cardioscope.models import CardioNet, count_parameters
from cardioscope.training import get_device, train_model, predict_logits, save_checkpoint, load_checkpoint
from cardioscope.metrics import per_class, summary
pd.set_option("display.width", 200)
DEVICE = get_device()
EPOCHS = int(os.environ.get("CARDIOSCOPE_EPOCHS", 12))
gpu = f" ({torch.cuda.get_device_name(0)})" if DEVICE.type == "cuda" else " (no GPU: training takes ~3 min per model)"
print(f"PyTorch {torch.__version__} | device: {DEVICE}{gpu}")
from cardioscope.metrics import ece, fit_temperature, softmax, triage
from cardioscope.explain import grad_cam, rhythm_counterfactual
from cardioscope.holter import analyze_ecg, compare_with_reference, format_report

## Catch-up: data and models

Your own models from this morning are used if they are still there; otherwise the pretrained ones shipped with the workshop.

In [ ]:
D.download_mitdb()
data = D.build_dataset()
ds1, ds2 = D.by_records(data, C.DS1), D.by_records(data, C.DS2)
train_set, val_set = D.holdout_split(ds1, frac=0.1, seed=0)
cal, ev = D.by_records(data, C.CAL_RECORDS), D.by_records(data, C.EVAL_RECORDS)

def load(name):
    mine = f"checkpoints/my_cardionet_{name}.pt"
    path = mine if os.path.exists(mine) else f"checkpoints/cardionet_{name}.pt"
    model, _ = load_checkpoint(path, DEVICE)
    print("loaded", path)
    return model

model_m, model_r, model_c = load("morph"), load("rhythm"), load("context")

## Lab 5. An AI that knows when it doesn't know

When the model says "V with probability 0.95", is it right 95 % of the time? The gap between confidence and accuracy is
the **Expected Calibration Error (ECE)**. *Temperature scaling* (Guo et al., ICML 2017) fixes it with one number T:
probabilities = softmax(logits / T).

We split the 22 unseen patients in two cohorts: 11 **calibration** patients (to fit T) and 11 **evaluation** patients.

In [ ]:
cache, rows = {}, {}
for name, m in (("morphology", model_m), ("+ rhythm", model_r), ("patient context", model_c)):
    lv, lc, le = (predict_logits(m, d) for d in (val_set, cal, ev))
    t_same, t_new = fit_temperature(lv, val_set["y"]), fit_temperature(lc, cal["y"])
    cache[name] = {"logits_eval": le, "T": t_new,
                   "gain": ece(softmax(le), ev["y"]) - ece(softmax(le, t_new), ev["y"])}
    rows[name] = {"ECE, no calibration (%)": 100 * ece(softmax(le), ev["y"]),
                  "T fitted on same patients": t_same,
                  "ECE with that T (%)": 100 * ece(softmax(le, t_same), ev["y"]),
                  "T fitted on new patients": t_new,
                  "ECE with this T (%)": 100 * ece(softmax(le, t_new), ev["y"])}
pd.DataFrame(rows).T.round(2)

In [ ]:
name = max(cache, key=lambda k: cache[k]["gain"])        # the most striking case
le, T_new = cache[name]["logits_eval"], cache[name]["T"]
viz.plot_reliability({f"model '{name}', T = 1": softmax(le),
                      f"T = {T_new:.2f} fitted on new patients": softmax(le, T_new)}, ev["y"])

**Lesson:** on the patients it was trained on, the network looks well calibrated (T close to 1). On new patients it is
**over-confident**, and only a calibration done on *new* patients reveals and corrects it. Calibration, like evaluation,
must be inter-patient.

### Triage: refer the uncertain beats to a cardiologist

In [ ]:
T = cache["patient context"]["T"]
probs_ev = softmax(cache["patient context"]["logits_eval"], T)
pd.DataFrame([triage(probs_ev, ev["y"], th) for th in (0.6, 0.8, 0.9, 0.95)])

In [ ]:
viz.plot_risk_coverage(probs_ev, ev["y"])

Beats referred to the cardiologist are several times more likely to be wrong than the beats the model keeps.
**Your turn:** which threshold would you choose for a screening tool? For a tool used at night without a doctor?

## Lab 6. Opening the black box

**Grad-CAM** colours the part of the beat that drove the decision (red = important).

In [ ]:
probs_ds2 = softmax(predict_logits(model_c, ds2), T)
pred = probs_ds2.argmax(1)
items = []
for c in ("N", "S", "V"):
    k = C.CLASSES.index(c)
    idx = np.flatnonzero((ds2["y"] == k) & (pred == k))
    i = idx[np.argmax(probs_ds2[idx, k])]
    cam, p, _ = grad_cam(model_c, ds2["X"][i], ds2["D"][i], ds2["rr"][i], temperature=T)
    items.append({"beat": D.center_crop(ds2["X"][i]), "cam": cam, "probs": p,
                  "title": f"Record {ds2['rec'][i]}: a {c} beat"})
viz.plot_gradcam(items)

**Counterfactual question:** what if the premature beat had arrived exactly on time? If the probability of S collapses,
the decision was driven by **timing**, exactly like a cardiologist's.

In [ ]:
k = C.CLASSES.index("S")
idx = np.flatnonzero((ds2["y"] == k) & (pred == k))[:8]
rows = []
for i in idx:
    actual, on_time = rhythm_counterfactual(model_c, ds2["X"][i], ds2["D"][i], ds2["rr"][i], temperature=T)
    rows.append({"record": int(ds2["rec"][i]), "RR before / median RR": round(float(ds2["rr"][i, 0]), 2),
                 "p(S), real beat": round(float(actual[k]), 2), "p(S), if on time": round(float(on_time[k]), 2)})
pd.DataFrame(rows)

## Lab 7. From beats to patient risk: an AI Holter

A real Holter has no annotations. The full pipeline (`cardioscope/holter.py`):

1. automatic QRS detection (XQRS algorithm),
2. classification of every beat, with a confidence,
3. clinical indicators: PVC burden, ventricular runs (non-sustained VT), couplets, bigeminy,
   supraventricular activity (Binici et al., Circulation 2010), pauses, simplified Lown grade.

We analyse four **unseen** patients and compare with the cardiologists. Watch record 219 (atrial fibrillation):
how many beats does the model refuse to decide on?

In [ ]:
reports = {}
for rec in (100, 233, 232, 219):
    sig, fs, samples, symbols = D.load_record(rec)
    res = analyze_ecg(sig, fs, model_c, temperature=T, threshold=0.8)
    ref = compare_with_reference(res, samples, symbols, fs)
    reports[rec] = (res, ref)
    print(f"=== Record {rec} | automatic QRS detection: Se {100 * ref['qrs_se']:.1f} %, PPV {100 * ref['qrs_ppv']:.1f} %")
    print(format_report(res.report), "\n")

In [ ]:
res, ref = reports[233]
first_v = res.positions[res.labels == C.CLASSES.index("V")][0] / res.fs
viz.plot_holter_strip(res, first_v - 3, 10, ref, "Record 233: AI labels (top) vs cardiologists (bottom)")

In [ ]:
per_class(ref["y_true"], ref["y_pred"])

In [ ]:
viz.plot_tachogram(reports[232][0], "Record 232: RR tachogram, pauses appear above the dashed line")

## Deploy CardioScope

One cell turns the model into a web application. On Colab, open the public `*.gradio.live` link on your phone.

In [ ]:
from app import build_app, launch
demo = build_app("checkpoints/cardionet_context.pt")   # or your own: "checkpoints/my_cardionet_context.pt"
launch(demo, share=IN_COLAB, prevent_thread_lock=True)

## Discussion: responsible AI in cardiology (15 min)

1. **Data.** 47 patients, Boston, 1975-1979, 30-minute recordings, lead MLII. Would the model work on a Tunisian
   population? On a smartwatch (lead I)? On a 12-lead ECG? How would you check?
2. **Blind spots.** No atrial fibrillation class, paced beats excluded, noisy segments. What happens with record 219?
3. **Regulation.** Software that detects arrhythmias for diagnosis is a medical device (EU MDR) and a high-risk AI system
   under the EU AI Act. In Tunisia, health data processing falls under organic law 2004-63 on personal data protection
   and the supervision of the INPDP.
4. **Human in the loop.** Who is responsible for a missed ventricular tachycardia? Who chooses the referral threshold?
5. **Next steps.** 12-lead deep learning (PTB-XL), external validation on other databases, prospective study.

*CardioScope is teaching material, not a medical device.*